# Revisão de Programação C — confira suas previsões

**Programação C (COMP0512) — UFS — 2026.2**

Na aula você respondeu oito exercícios no papel, sem compilar nada. Aqui estão os mesmos
programas, prontos para rodar. Para cada rodada:

1. releia a pergunta e a resposta que você escreveu na folha;
2. execute as células;
3. se a saída discordar da sua previsão, volte ao resumo daquela rodada nos slides.

Os números de `sizeof` supõem uma máquina de 64 bits comum (`char` 1, `int` 4, `double` 8,
ponteiro 8) — é o caso do Colab.

In [ ]:
# Confira se o gcc está disponível no seu ambiente
!gcc --version | head -1

## R1. Tipos e registros

Em que offset começa cada campo de `struct Leitura`? Quanto vale o `sizeof`? E depois de
reordenar os campos?

`offsetof` (de `<stddef.h>`) devolve o deslocamento de um campo em relação ao início do registro.

In [ ]:
%%writefile r1_registro.c
/* r1_registro.c --- quantos bytes ocupa cada registro? */
#include <stdio.h>
#include <stddef.h>

struct Leitura {
    char sensor;
    double valor;
    int codigo;
};

struct LeituraReordenada {
    double valor;
    int codigo;
    char sensor;
};

int main(void)
{
    printf("Leitura: %zu bytes (valor em %zu, codigo em %zu)\n",
           sizeof(struct Leitura), offsetof(struct Leitura, valor),
           offsetof(struct Leitura, codigo));
    printf("LeituraReordenada: %zu bytes\n", sizeof(struct LeituraReordenada));
    return 0;
}

In [ ]:
!gcc -Wall -Wextra r1_registro.c -o r1 && ./r1

**Tarefa.** Acrescente um campo `char status;` logo depois de `sensor` em `struct Leitura`,
preveja o novo `sizeof` e rode de novo. Mudou? Por quê?

## R2. `enum` e `typedef`

Quanto valem `AZUL` e `NUM_CORES`? `enum Cor c = 42;` compila? E `b = a;` com `Nome`?

In [ ]:
%%writefile r2_enum.c
/* r2_enum.c --- que numeros o compilador atribuiu? */
#include <stdio.h>

enum Cor { VERMELHO, VERDE = 5, AZUL, NUM_CORES };

int main(void)
{
    enum Cor c = 42;            /* (b) compila? */
    printf("%d %d %d %d\n", VERMELHO, VERDE, AZUL, NUM_CORES);
    printf("c = %d\n", c);
    return 0;
}

In [ ]:
!gcc -Wall -Wextra r2_enum.c -o r2 && ./r2

A próxima compilação **deve falhar** — leia a mensagem do compilador.

In [ ]:
%%writefile r2_typedef.c
/* r2_typedef.c --- (c) compila? */
typedef char Nome[20];

int main(void)
{
    Nome a = "Ana", b;
    b = a;
    return 0;
}

In [ ]:
!gcc -Wall -Wextra r2_typedef.c -o r2t

**Tarefa.** Corrija `r2_typedef.c` com `strcpy` (inclua `<string.h>`) e imprima `b`.

## R3. Ponteiros

Para onde `p` aponta depois de `p += 2`? O que o programa imprime?

In [ ]:
%%writefile r3_ponteiros.c
/* r3_ponteiros.c --- rastreie antes de rodar */
#include <stdio.h>

int main(void)
{
    int v[] = { 10, 20, 30, 40 };
    int *p = v + 1;
    int x = 5;
    int *q = &x;

    *q = *p + 1;
    p += 2;
    *p = x * 2;

    printf("%d %d %d %td\n", x, v[3], *(v + 2), p - v);
    return 0;
}

In [ ]:
!gcc -Wall -Wextra r3_ponteiros.c -o r3 && ./r3

**Tarefa.** Troque `p += 2;` por `p--;` e preveja a nova saída antes de rodar. Qual elemento
do vetor muda agora?

## R4. Vetores e strings

O que aparece nas linhas `A:`, `B:` e `C:`? Repare no aviso que o compilador dá para `f`.

In [ ]:
%%writefile r4_vetores.c
/* r4_vetores.c --- o que se perde ao entrar na funcao */
#include <stdio.h>
#include <string.h>

void f(int v[10])
{
    printf("B: %zu\n", sizeof(v));
}

int main(void)
{
    int a[10];
    char s[20] = "prova";

    printf("A: %zu %zu %zu\n", sizeof(a), sizeof(s), strlen(s));
    f(a);
    s[2] = '\0';
    printf("C: %s %zu\n", s, strlen(s));
    return 0;
}

In [ ]:
!gcc -Wall -Wextra r4_vetores.c -o r4 && ./r4

**Tarefa.** Mude `f` para `void f(int v[], int n)`, imprimindo `n * sizeof(v[0])`, e chame
`f(a, 10)`. Agora a linha `B:` bate com `sizeof(a)`?

## R5. Alocação dinâmica

Este programa compila **sem nenhum aviso** com `-Wall -Wextra` e tem quatro falhas. Rode primeiro
do jeito normal — pode até parecer que funciona — e depois com o AddressSanitizer, que aponta a
linha do primeiro problema.

In [ ]:
%%writefile r5_alocacao.c
/* r5_alocacao.c --- compila sem aviso, mas tem quatro falhas */
#include <stdio.h>
#include <stdlib.h>

int *cria(int n)
{
    int *v = malloc(n);
    for (int i = 0; i < n; i++)
        v[i] = i * i;
    return v;
}

int main(void)
{
    int *v = cria(5);
    v = realloc(v, 10 * sizeof(int));
    free(v);
    printf("%d\n", v[4]);
    return 0;
}

In [ ]:
!gcc -Wall -Wextra r5_alocacao.c -o r5 && ./r5

In [ ]:
!gcc -g -fsanitize=address r5_alocacao.c -o r5a && ./r5a 2>&1 | head -20

A versão corrigida, com as quatro falhas resolvidas, roda limpa sob o sanitizer:

In [ ]:
%%writefile r5_alocacao_corrigido.c
/* r5_alocacao_corrigido.c --- as quatro falhas corrigidas */
#include <stdio.h>
#include <stdlib.h>

int *cria(int n)
{
    int *v = malloc(n * sizeof(int));
    if (v == NULL)
        return NULL;
    for (int i = 0; i < n; i++)
        v[i] = i * i;
    return v;
}

int main(void)
{
    int *v = cria(5);
    if (v == NULL)
        return 1;
    int *tmp = realloc(v, 10 * sizeof(int));
    if (tmp == NULL) {
        free(v);
        return 1;
    }
    v = tmp;
    printf("%d\n", v[4]);
    free(v);
    v = NULL;
    return 0;
}

In [ ]:
!gcc -g -Wall -Wextra -fsanitize=address r5_alocacao_corrigido.c -o r5c && ./r5c

## R6. Matrizes dinâmicas

O mesmo 3×4 nos dois formatos: onde fica o elemento (2, 1) no bloco único, e quantos `malloc`
cada versão faz.

In [ ]:
%%writefile r6_matriz.c
/* r6_matriz.c --- a mesma matriz 3x4 nos dois formatos */
#include <stdio.h>
#include <stdlib.h>

int main(void)
{
    int nl = 3, nc = 4, chamadas = 0;

    double *b = malloc((size_t)nl * nc * sizeof(double));
    chamadas++;
    for (int k = 0; k < nl * nc; k++)
        b[k] = k;
    printf("bloco unico: elemento (2,1) no indice %d, valor %.0f; %d malloc\n",
           2 * nc + 1, b[2 * nc + 1], chamadas);
    free(b);

    chamadas = 0;
    double **m = malloc(nl * sizeof(double *));
    chamadas++;
    for (int i = 0; i < nl; i++) {
        m[i] = malloc(nc * sizeof(double));
        chamadas++;
    }
    printf("double **: %d malloc\n", chamadas);
    for (int i = 0; i < nl; i++)            /* de dentro para fora */
        free(m[i]);
    free(m);
    return 0;
}

In [ ]:
!gcc -g -Wall -Wextra -fsanitize=address r6_matriz.c -o r6 && ./r6

**Tarefa.** Inverta a liberação da versão `double **` (primeiro `free(m)`, depois as linhas) e
rode de novo com `-fsanitize=address`. Que erro aparece, e em que linha?

## R7. Modularização

Os quatro arquivos do exercício. Rode os três comandos na ordem e compare cada mensagem com a sua
resposta. As mensagens abaixo são as do `gcc`/`ld` no Linux; no macOS o texto muda, mas o erro é o
mesmo.

In [ ]:
%%writefile estat.h
struct Resumo {
    double media;
    double maior;
};

double estat_media(const double *v, int n);

In [ ]:
%%writefile estat.c
#include "estat.h"

double estat_media(const double *v, int n)
{
    double soma = 0.0;
    for (int i = 0; i < n; i++)
        soma += v[i];
    return soma / n;
}

void troca(double *a, double *b)
{
    double t = *a;
    *a = *b;
    *b = t;
}

In [ ]:
%%writefile turma.h
#include "estat.h"

struct Resumo turma_resume(const double *v, int n);

In [ ]:
%%writefile turma.c
#include "turma.h"

void troca(double *a, double *b)
{
    double t = *a;
    *a = *b;
    *b = t;
}

struct Resumo turma_resume(const double *v, int n)
{
    struct Resumo r = { estat_media(v, n), v[0] };
    for (int i = 1; i < n; i++)
        if (v[i] > r.maior)
            r.maior = v[i];
    return r;
}

In [ ]:
%%writefile main.c
#include <stdio.h>
#include "estat.h"
#include "turma.h"

int main(void)
{
    double notas[] = { 7.5, 9.0, 4.0 };
    struct Resumo r = turma_resume(notas, 3);
    printf("media %.2f, maior %.1f\n", r.media, r.maior);
    return 0;
}

In [ ]:
# (a) so compilar main.c: erro de compilacao
!gcc -Wall -c main.c

Para seguir, acrescente a guarda de inclusão em `estat.h` e `turma.h` (a célula abaixo faz isso).

In [ ]:
!printf '#ifndef ESTAT_H\n#define ESTAT_H\n' | cat - estat.h > tmp && printf '#endif\n' >> tmp && mv tmp estat.h
!printf '#ifndef TURMA_H\n#define TURMA_H\n' | cat - turma.h > tmp && printf '#endif\n' >> tmp && mv tmp turma.h
!cat estat.h

In [ ]:
# (b) faltou estat.c: erro de ligacao
!gcc -Wall main.c turma.c -o notas

In [ ]:
# (c) troca definida em dois .c: erro de ligacao
!gcc -Wall main.c turma.c estat.c -o notas

**Tarefa.** Marque as duas `troca` como `static`, recompile o comando (c) e execute `./notas`.

## R8. Padrões de codificação

As duas versões imprimem a mesma coisa. A diferença está em quem consegue ler, reusar e testar.

In [ ]:
%%writefile r8_ranking.c
/* r8_ranking.c --- funciona, mas nao passa na revisao */
#include <stdio.h>

int n, k;
double R[50];

void calc(double *v)
{
    k = 0;
    for (int i = 0; i < n; i++)
        if (v[i] >= 7.0)
            R[k++] = v[i];
}

int main(void)
{
    double medias[] = { 8.0, 5.5, 7.0, 9.5 };
    n = 4;
    calc(medias);
    for (int i = 0; i < k; i++)
        printf("%.1f ", R[i]);
    printf("\n");
    return 0;
}

In [ ]:
!gcc -Wall -Wextra r8_ranking.c -o r8 && ./r8

In [ ]:
%%writefile r8_ranking_corrigido.c
/* r8_ranking_corrigido.c --- o mesmo calculo dentro do padrao */
#include <stdio.h>

#define RANKING_MEDIA_APROVACAO 7.0

/* Copia para aprovados as medias >= RANKING_MEDIA_APROVACAO.
 * aprovados deve ter espaco para n elementos. Devolve quantas copiou. */
static int ranking_filtra_aprovados(const double *medias, int n,
                                    double *aprovados)
{
    int qtd = 0;
    for (int i = 0; i < n; i++)
        if (medias[i] >= RANKING_MEDIA_APROVACAO)
            aprovados[qtd++] = medias[i];
    return qtd;
}

int main(void)
{
    const double medias[] = { 8.0, 5.5, 7.0, 9.5 };
    const int n = (int)(sizeof medias / sizeof medias[0]);
    double aprovados[sizeof medias / sizeof medias[0]];

    int qtd = ranking_filtra_aprovados(medias, n, aprovados);
    for (int i = 0; i < qtd; i++)
        printf("%.1f ", aprovados[i]);
    printf("\n");
    return 0;
}

In [ ]:
!gcc -Wall -Wextra r8_ranking_corrigido.c -o r8c && ./r8c

In [ ]:
# as duas saidas devem ser identicas
!./r8 > saida_antes.txt && ./r8c > saida_depois.txt && diff saida_antes.txt saida_depois.txt && echo 'saidas identicas'

## Desafio

Junte R5, R6 e R8: escreva um módulo `turma.h`/`turma.c`, dentro do padrão da rodada 8, com

- `double *turma_cria(int nl, int nc)` — matriz em bloco único, devolvendo `NULL` na falha;
- `double turma_media_linha(const double *m, int nc, int i)`;
- `void turma_libera(double *m)`.

Compile com `-Wall -Wextra -fsanitize=address` e mostre, num `main.c`, a média de cada aluno de
uma turma 3×4. Nenhum aviso, nenhum erro do sanitizer.

In [ ]:
# Sua solucao aqui (use %%writefile para cada arquivo)

## Referências

Veja o arquivo `../referencias.bib`. Os slides indicam, em cada resumo, a seção do livro de onde vem
o conceito.